# Notebook 1 (simple, testable): recover from a failed or timed-out tool call

**Outcome:** the agent completes a multi-step task and recovers cleanly from a failed or timed-out tool call.

Plain Python, no agent framework: a `while`-style tool-calling loop, like the "agent loop from scratch" lessons.

**Task:** flight, then hotel, then total cost for a 3-night trip.
**Injected faults:**
- `search_flights` hangs on its first call, so the **timeout + retry** layer fixes it silently.
- `search_hotels_primary` always fails, so the model must read the error and call `search_hotels_backup`.

## 0. Setup (pick one)

**Option A: no model at all (`BACKEND = "mock"`).** A scripted fake LLM drives the agent loop. This tests *your harness* (retries, pause/resume, memory) deterministically. Start here.

**Option B: free open-source model via Ollama (`BACKEND = "ollama"`).**
1. Install Ollama: https://ollama.com
2. `ollama pull qwen2.5:7b` (or `llama3.1:8b`; the model must support tool calling)
3. Keep Ollama running (it serves an OpenAI-compatible API at `http://localhost:11434/v1`)

Any other OpenAI-compatible free endpoint (Groq free tier, OpenRouter free models, LM Studio, vLLM) works by changing `base_url`, `api_key` and `MODEL`.

In [2]:
%pip install -q openai
import json, time, concurrent.futures as cf
from openai import OpenAI

BACKEND = "mock"        # "mock" or "ollama"
MODEL = "qwen2.5:7b"
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

## 1. Tools with timeout and retry

In [3]:
LOG = []                       # (tool, attempt, outcome): your evidence trail
_state = {"flight_calls": 0}
_pool = cf.ThreadPoolExecutor(max_workers=4)

def reset():
    LOG.clear(); _state["flight_calls"] = 0

def with_retry(fn, timeout_s=1.0, retries=2, backoff_s=0.2):
    """Run fn in a thread with a timeout; retry; on total failure return a structured error (never raise)."""
    def wrapper(**kwargs):
        last = None
        for attempt in range(1, retries + 2):
            fut = _pool.submit(fn, **kwargs)
            try:
                res = fut.result(timeout=timeout_s)
                LOG.append((fn.__name__, attempt, "ok"))
                return res
            except cf.TimeoutError:
                last = f"timed out after {timeout_s}s"
            except Exception as e:
                last = f"{type(e).__name__}: {e}"
            LOG.append((fn.__name__, attempt, f"FAILED ({last})"))
            time.sleep(backoff_s * attempt)
        return {"status": "error", "tool": fn.__name__, "error": last, "attempts": retries + 1,
                "hint": "Do not call this tool again. Use a fallback tool if one exists."}
    wrapper.__name__ = fn.__name__
    return wrapper

def search_flights(origin, destination):
    _state["flight_calls"] += 1
    if _state["flight_calls"] == 1:
        time.sleep(3)                      # simulated hang
    return {"status": "ok", "route": f"{origin}->{destination}", "price_usd": 420.0}

def search_hotels_primary(city, nights):
    raise ConnectionError("503 from primary hotel provider")   # always down

def search_hotels_backup(city, nights):
    return {"status": "ok", "city": city, "price_per_night_usd": 150.0, "nights": nights}

def calculate_total(flight_usd, hotel_per_night_usd, nights):
    return {"status": "ok", "total_usd": float(flight_usd) + float(hotel_per_night_usd) * int(nights)}

TOOLS = {
    "search_flights": with_retry(search_flights, timeout_s=1.0, retries=2),
    "search_hotels_primary": with_retry(search_hotels_primary, retries=1),
    "search_hotels_backup": with_retry(search_hotels_backup, retries=1),
    "calculate_total": with_retry(calculate_total, retries=0),
}

def fn_schema(name, desc, props, required):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": required}}}

TOOL_SCHEMAS = [
    fn_schema("search_flights", "Find the cheapest flight between two IATA airport codes.",
              {"origin": {"type": "string"}, "destination": {"type": "string"}}, ["origin", "destination"]),
    fn_schema("search_hotels_primary", "Search the primary hotel provider.",
              {"city": {"type": "string"}, "nights": {"type": "integer"}}, ["city", "nights"]),
    fn_schema("search_hotels_backup", "Backup hotel provider. Use ONLY if the primary provider returned an error.",
              {"city": {"type": "string"}, "nights": {"type": "integer"}}, ["city", "nights"]),
    fn_schema("calculate_total", "Compute total trip cost in USD.",
              {"flight_usd": {"type": "number"}, "hotel_per_night_usd": {"type": "number"}, "nights": {"type": "integer"}},
              ["flight_usd", "hotel_per_night_usd", "nights"]),
]

SYSTEM = """You are a trip planner. Steps: search_flights, then search_hotels_primary, then calculate_total, then answer.
If a tool returns status "error": never retry it. If search_hotels_primary fails, call search_hotels_backup.
In the final answer give the total and say which step failed and how you recovered. Never invent prices."""

## 2. The agent loop, and a scripted mock LLM

In [4]:
def assistant_msg(content, calls):
    return {"role": "assistant", "content": content,
            "tool_calls": [{"id": c["id"], "type": "function",
                            "function": {"name": c["name"], "arguments": json.dumps(c["args"])}} for c in calls]}

def tool_msg(call_id, result):
    return {"role": "tool", "tool_call_id": call_id, "content": json.dumps(result)}

def history(messages):
    """[(tool_name, result_dict), ...] in order. Used by the mock LLM and by the tests."""
    id2name, out = {}, []
    for m in messages:
        if m["role"] == "assistant":
            for c in m.get("tool_calls") or []:
                id2name[c["id"]] = c["function"]["name"]
        elif m["role"] == "tool":
            out.append((id2name[m["tool_call_id"]], json.loads(m["content"])))
    return out

def ollama_llm(messages, tools):
    r = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
    m = r.choices[0].message
    calls = [{"id": c.id, "name": c.function.name, "args": json.loads(c.function.arguments or "{}")}
             for c in (m.tool_calls or [])]
    return m.content, calls


def mock_llm(messages, tools):
    """Deterministic stand-in for a model: follows the recovery policy we want the real model to follow."""
    h = history(messages); names = [n for n, _ in h]; res = dict(h)
    def call(name, args): return None, [{"id": f"mock-{len(h)}", "name": name, "args": args}]
    if "search_flights" not in names:
        return call("search_flights", {"origin": "YVR", "destination": "SFO"})
    if "search_hotels_primary" not in names:
        return call("search_hotels_primary", {"city": "San Francisco", "nights": 3})
    if res["search_hotels_primary"].get("status") == "error" and "search_hotels_backup" not in names:
        return call("search_hotels_backup", {"city": "San Francisco", "nights": 3})
    hotel = res.get("search_hotels_backup") or res["search_hotels_primary"]
    if "calculate_total" not in names:
        return call("calculate_total", {"flight_usd": res["search_flights"]["price_usd"],
                                         "hotel_per_night_usd": hotel["price_per_night_usd"], "nights": 3})
    total = res["calculate_total"]["total_usd"]
    return (f"Total: ${total:.0f} (flight $420 + 3 nights x $150). The primary hotel provider failed, "
            f"so I used the backup provider."), []

def get_llm():
    return {"mock": mock_llm, "ollama": ollama_llm}[BACKEND]

def run_agent(user_msg, max_steps=12):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
    for _ in range(max_steps):
        content, calls = get_llm()(messages, TOOL_SCHEMAS)
        if not calls:
            return content, messages
        messages.append(assistant_msg(content, calls))
        for c in calls:
            fn = TOOLS.get(c["name"])
            result = fn(**c["args"]) if fn else {"status": "error", "error": f"unknown tool {c['name']}"}
            print(f"  {c['name']}({c['args']}) -> {result}")
            messages.append(tool_msg(c["id"], result))
    raise RuntimeError("max_steps reached without a final answer")

## 3. Tests

In [5]:
def run_test(backend):
    global BACKEND
    BACKEND = backend
    reset()
    print(f"\n=== backend: {backend} ===")
    final, msgs = run_agent("Plan a 3-night trip from YVR to San Francisco.")
    print("\nFINAL:", final)
    print("\nAttempt log:")
    for t, a, o in LOG: print(f"  {t:<24} attempt {a}: {o}")

    checks = {
        "flight call timed out, then succeeded on retry":
            any(t == "search_flights" and "timed out" in o for t, a, o in LOG)
            and any(t == "search_flights" and o == "ok" for t, a, o in LOG),
        "primary hotel failed, backup used successfully":
            any(t == "search_hotels_primary" and o.startswith("FAILED") for t, a, o in LOG)
            and any(t == "search_hotels_backup" and o == "ok" for t, a, o in LOG),
        "final answer contains the correct total ($870)": bool(final) and "870" in final.replace(",", ""),
    }
    for name, ok in checks.items(): print(("PASS  " if ok else "FAIL  ") + name)
    return checks

# 1) Deterministic harness test: must pass 100%
assert all(run_test("mock").values())
print("\nOutcome met (mock): timeout retried, hard failure routed to fallback, task completed.")


=== backend: mock ===
  search_flights({'origin': 'YVR', 'destination': 'SFO'}) -> {'status': 'ok', 'route': 'YVR->SFO', 'price_usd': 420.0}
  search_hotels_primary({'city': 'San Francisco', 'nights': 3}) -> {'status': 'error', 'tool': 'search_hotels_primary', 'error': 'ConnectionError: 503 from primary hotel provider', 'attempts': 2, 'hint': 'Do not call this tool again. Use a fallback tool if one exists.'}
  search_hotels_backup({'city': 'San Francisco', 'nights': 3}) -> {'status': 'ok', 'city': 'San Francisco', 'price_per_night_usd': 150.0, 'nights': 3}
  calculate_total({'flight_usd': 420.0, 'hotel_per_night_usd': 150.0, 'nights': 3}) -> {'status': 'ok', 'total_usd': 870.0}

FINAL: Total: $870 (flight $420 + 3 nights x $150). The primary hotel provider failed, so I used the backup provider.

Attempt log:
  search_flights           attempt 1: FAILED (timed out after 1.0s)
  search_flights           attempt 2: ok
  search_hotels_primary    attempt 1: FAILED (ConnectionError: 503 fro

In [ ]:
# 2) Real open-source model via Ollama (needs Ollama running + model pulled).
#    Small models can be flaky, so this prints PASS/FAIL rather than asserting. Re-run if needed.
try:
    run_test("ollama")
except Exception as e:
    print("Ollama run failed:", type(e).__name__, e)